# Modelling

# Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import statsmodels.formula.api as smf
from sklearn.metrics import f1_score, accuracy_score

In [2]:
train_df = pd.read_csv("../../data/processed/train.csv", keep_default_na=False)
test_df = pd.read_csv("../../data/processed/test.csv", keep_default_na=False)
val_df = pd.read_csv("../../data/processed/val.csv", keep_default_na=False)

In [3]:
train_df.shape, test_df.shape, val_df.shape

((2051, 95), (440, 95), (439, 95))

# Linear Regression Models

In [ ]:
def build_regression_model(formula, train_df, val_df, model_number):
    
    model = smf.ols( 
        formula, 
        data=train_df
    ).fit()

    print(model.summary())

    predictions = model.predict(val_df)
    rmse = np.sqrt(mean_squared_error(val_df["Log_SalePrice"], predictions))

    predictions_original_scale = np.exp(predictions)
    rmse_original_scale = np.sqrt(mean_squared_error(np.exp(val_df["Log_SalePrice"]), predictions_original_scale))

    print(f"Model {model_number} - RMSE (Log Scale): {rmse:.4f}")
    print(f"Model {model_number} - RMSE (Original Scale): ${rmse_original_scale:,.0f}")
    

## Model 1

First model is a baseline with three numerical variables that had high Pearson correlation with Sale Price.

In [ ]:
formula1 = 'Log_SalePrice ~ Q("Overall Score") + Q("House Age") + TotalSF'

build_regression_model(formula1, train_df, val_df, model_number=1)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.803
Model:                            OLS   Adj. R-squared:                  0.802
Method:                 Least Squares   F-statistic:                     2776.
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        22:54:56   Log-Likelihood:                 618.13
No. Observations:                2051   AIC:                            -1228.
Df Residuals:                    2047   BIC:                            -1206.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             11.0646      0

The first model has a ${R^2}$ of 0.803, which means that 80.3% of the values can be explained by the three chosen features.

This shows that Prices are highly influenced by the quality and sizing of a property but also shows that more recent houses tend to be more expensive.

If we take a look at the House Age coefficient we can also see that it is negative which shows that the price of a housing goes down as it ages.

## Model 2

As we've seen, house age has an impact on the model's performance, In this iteration I will see if `Years Since Remodel` has a significant impact on the performance as well.

By looking at the features with more correlation with SalePrice, we can see that the Garage grouping makes a difference, so I will be testing the `Garage Score` variable as well.

In [ ]:
formula2 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")'

build_regression_model(formula2, train_df, val_df, model_number=2)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.814
Model:                            OLS   Adj. R-squared:                  0.814
Method:                 Least Squares   F-statistic:                     1793.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:34:08   Log-Likelihood:                 680.04
No. Observations:                2051   AIC:                            -1348.
Df Residuals:                    2045   BIC:                            -1314.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

As we can see the second model obtained an $R^2$ of 0.814, which means 81.4% of the values can be explained by the current features. This represent a 1.1% improvement of our model with the addition of only two extra features.

As expected, `Garage Score` coefficient indicated that it has a positive impact on the `Sale Price` value, i.e houses with better Garages are more expensive.

On the opposite direction, the `Years Since Remodel` behaves like House Age. The negative coefficient indicates that the longer it has been since a house was renovated, the cheaper the property is going to be. 

## Model 3

In this try of the model I want to verify how the model performs if I add some of the highest correlated categorical features: `Neighbourhood` and `Condition 1` and `Condition 2`.

A good and safe Neighbourhood in a nice location (near schools, hospitals, etc), is more likely to be more expensive.

On the same thought process the features `Condition 1` and `Condition 2` mean a certain proximity of the property to various conditions, which can add value to the property.

In [ ]:
formula3 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood' \
                        ' + Q("Condition 1")' \
                        ' + Q("Condition 2")' \

build_regression_model(formula3, train_df, val_df, model_number=3)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.850
Model:                            OLS   Adj. R-squared:                  0.846
Method:                 Least Squares   F-statistic:                     241.1
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:34:32   Log-Likelihood:                 897.65
No. Observations:                2051   AIC:                            -1699.
Df Residuals:                    2003   BIC:                            -1429.
Df Model:                          47                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

The third model obtained an $R^2$ of 0.850. Compared to the second model 81.4%, this is a 3,6% improvement of our model.

The coefficients on each categorical feature corroborate with the theory that being in different neighbourhoods or having proximity to certain conditions can make the price of the house be lower or higher.

## Model 4

In this model I will test the impact of `Has Fireplace` and `Total Bath`. 

Usually, in property listings, houses with a greater amount of bathroom mean that they are bigger and therefore more expensive.

Having a Fireplace might also be an indicator of higher priced properties as it is seen as a luxury item.

In [ ]:
formula4 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood' \
                        ' + Q("Condition 1")' \
                        ' + Q("Condition 2")' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \

build_regression_model(formula4, train_df, val_df, model_number=4)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.864
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     260.1
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:34:49   Log-Likelihood:                 1001.8
No. Observations:                2051   AIC:                            -1904.
Df Residuals:                    2001   BIC:                            -1622.
Df Model:                          49                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

From the 3rd to the 4th model there was an improvement of 1.4% of the value of $R^2$ (was 0.850, now 0.864).

Analyzing the coefficients of the newly added features, we can confirm that both of them have a positive impact on the price range i.e if a house has a fireplace, or a bigger amount of bathrooms, the price will be bigger.

## Model 5

`Mas Vnr Area` and `Mas Vnr Type` also seem to have some sort of impact in the value of Sale Price accordingly to the correlation analysis, let's see if that translates to an improvement of our model.

In [ ]:
formula5 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood' \
                        ' + Q("Condition 1")' \
                        ' + Q("Condition 2")' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("Mas Vnr Area")' \
                        ' + Q("Mas Vnr Type")' \

build_regression_model(formula5, train_df, val_df, model_number=5)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     236.3
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:35:01   Log-Likelihood:                 1005.3
No. Observations:                2051   AIC:                            -1901.
Df Residuals:                    1996   BIC:                            -1591.
Df Model:                          54                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

Altough the model improved by 0.1% the features don't seem to have a significant impact on the model, especially with Mas Vnr Area having a p_value of 0.696.

## Model 6

in this model I have decided to test the `MS SubClass` and `MS Zoning` feature alongside `Mas Vnr Type` (that altough didn't have explendid results was still able to improve the model slightly).

`MS SubClass` might raise or lower the property price since different dwellings might be more appealing or not.

`MS Zoning` refers to the zone of the property. A house in an industrial zone might not be as appealing as one in a Residential Area.

In [ ]:
formula6 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood' \
                        ' + Q("Condition 1")' \
                        ' + Q("Condition 2")' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("Mas Vnr Type")' \
                        ' + Q("MS SubClass")' \
                        ' + Q("MS Zoning")' \

build_regression_model(formula6, train_df, val_df, model_number=6)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.870
Model:                            OLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                     222.8
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:35:09   Log-Likelihood:                 1049.3
No. Observations:                2051   AIC:                            -1977.
Df Residuals:                    1990   BIC:                            -1633.
Df Model:                          60                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

The 5th model has a 0.870 $R^2$, the greatest so far. The addition of `MS SubClass` and `MS Zoning` greatly improved the model as expected, with different zones and types of property having different impacts on the price of the house.

## Model 7 

For this last attempt I will see if `Sale Condition` has an impact on the model.

A Sale Condition of 'Partial' means that the house was not completed when the house was sold, which means the house is not yet fully built and therefore is very new, which can make it more expensive.

In [ ]:
formula7 = 'Log_SalePrice ~ Q("Overall Score")' \
                        ' + Q("House Age")' \
                        ' + TotalSF' \
                        ' + Q("Years Since Remodel") ' \
                        ' + Q("Garage Score")' \
                        ' + Neighborhood' \
                        ' + Q("Condition 1")' \
                        ' + Q("Condition 2")' \
                        ' + Q("Has Fireplace")' \
                        ' + Q("Total Bath")' \
                        ' + Q("Mas Vnr Type")' \
                        ' + Q("MS SubClass")' \
                        ' + Q("MS Zoning")' \
                        ' + Q("Sale Condition")' \

build_regression_model(formula7, train_df, val_df, model_number=7)

                            OLS Regression Results                            
Dep. Variable:          Log_SalePrice   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     211.5
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:35:16   Log-Likelihood:                 1076.5
No. Observations:                2051   AIC:                            -2021.
Df Residuals:                    1985   BIC:                            -1650.
Df Model:                          65                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

This last model proves to be the best with an %R^2% of 0.874 (87,4% of values are explained by the features chosen).

As expected, the `Sale Condition` feature significantly improved the model meaning that different conditions during the sale have different impacts on the Sale Price.

The RMSE in the original scale being 23,619 is an acceptable error considering how big house pricings can be.

# Logistic Regression